# Analyze per Die Matrix

This notebook demonstrates how to find the **reference behavior** for each die matrix, compare individual pieces against it, and identify which process segments show the most variability.

All analysis is performed on the gold parquet dataset (clean, validated pieces with partial times).

In [1]:
import pandas as pd
import numpy as np
import altair as alt
from pathlib import Path

GOLD_FILE = Path('../data/gold/pieces.parquet')
df = pd.read_parquet(GOLD_FILE)
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
df['date'] = df['timestamp'].dt.date

PARTIAL_COLS = [
    'partial_furnace_to_2nd_strike_s',
    'partial_2nd_to_3rd_strike_s',
    'partial_3rd_to_4th_strike_s',
    'partial_4th_strike_to_auxiliary_press_s',
    'partial_auxiliary_press_to_bath_s',
]
PARTIAL_LABELS = {
    'partial_furnace_to_2nd_strike_s':          'Furnace→2nd',
    'partial_2nd_to_3rd_strike_s':              '2nd→3rd',
    'partial_3rd_to_4th_strike_s':              '3rd→4th',
    'partial_4th_strike_to_auxiliary_press_s':  '4th→Aux',
    'partial_auxiliary_press_to_bath_s':        'Aux→Bath',
}
CUMULATIVE_COLS = [
    'lifetime_2nd_strike_s', 'lifetime_3rd_strike_s',
    'lifetime_4th_strike_s', 'lifetime_auxiliary_press_s', 'lifetime_bath_s',
]

pd.set_option('display.float_format', '{:.2f}'.format)
print(f'Loaded {len(df):,} pieces from gold parquet')
print(f'Matrices: {sorted(df["die_matrix"].unique().tolist())}')


Loaded 167,679 pieces from gold parquet
Matrices: [4974, 5052, 5090, 5091]


## 1. Reference profile per die matrix

The **reference (optimal) behavior** for each die matrix is defined by its median cumulative times. The median is more robust than the mean — it is not pulled by residual edge cases.

This table shows how long a typical piece takes to reach each stage, per matrix.

In [2]:
ref_cum = df.groupby('die_matrix')[CUMULATIVE_COLS].median().round(1)
ref_cum.columns = ['2nd strike', '3rd strike', '4th strike', 'Aux press', 'Bath']
display(ref_cum)
print('\nReference = median cumulative time (s) from furnace exit. Robust to residual edge cases.')


,2nd strike,3rd strike,4th strike,Aux press,Bath
die_matrix,,,,,
4974,17.30,23.90,37.10,54.20,56.00
5052,18.30,25.30,39.30,56.70,58.30
5090,17.70,24.60,38.50,56.50,58.10
5091,18.50,25.60,38.20,57.50,59.10



Reference = median cumulative time (s) from furnace exit. Robust to residual edge cases.


## 2. Reference partial times per die matrix

The partial times (time spent **between** consecutive stages) are more diagnostically useful than cumulative times — they isolate each segment of the process.

In [3]:
ref_partial = df.groupby('die_matrix')[PARTIAL_COLS].median().round(1)
ref_partial.columns = list(PARTIAL_LABELS.values())
display(ref_partial)
print('\nPartial times isolate each segment — key diagnostic values for delay attribution.')
print('Aux→Bath is the shortest segment (~1.5-2s); Furnace→2nd is the longest (~18s).')


,Furnace→2nd,2nd→3rd,3rd→4th,4th→Aux,Aux→Bath
die_matrix,,,,,
4974,17.30,6.50,13.10,17.00,1.80
5052,18.30,6.90,13.70,17.30,1.60
5090,17.70,6.80,13.80,17.70,1.60
5091,18.50,7.00,13.50,17.00,1.60



Partial times isolate each segment — key diagnostic values for delay attribution.
Aux→Bath is the shortest segment (~1.5-2s); Furnace→2nd is the longest (~18s).


## 3. Variability per segment per die matrix

Which segments are most variable? High standard deviation relative to the median (coefficient of variation) indicates segments where the process is less stable — these are the primary candidates for delay investigation.

**CV = std / median × 100%** — higher CV means more variability relative to the typical value.

In [4]:
cv_rows = []
for matrix, grp in df.groupby('die_matrix'):
    for col in PARTIAL_COLS:
        vals = grp[col].dropna()
        if len(vals) < 10: continue
        median = vals.median()
        std    = vals.std()
        cv     = 100 * std / median if median > 0 else np.nan
        cv_rows.append({'die_matrix': matrix, 'segment': PARTIAL_LABELS[col],
                        'median_s': round(median, 1), 'std_s': round(std, 2), 'CV_%': round(cv, 1)})

cv_df = pd.DataFrame(cv_rows)
cv_pivot = cv_df.pivot(index='segment', columns='die_matrix', values='CV_%')
display(cv_pivot.round(1))
print('\nHigher CV = more variable segment = primary candidate for process improvement.')
most_variable = cv_df.groupby('segment')['CV_%'].mean().idxmax()
print(f'Most variable segment overall: {most_variable}')


die_matrix,4974,5052,5090,5091
segment,,,,
2nd→3rd,4.10,7.30,10.90,10.50
3rd→4th,2.40,6.20,8.10,9.10
4th→Aux,5.20,6.60,9.10,6.10
Aux→Bath,2.70,3.10,5.40,5.30
Furnace→2nd,9.10,10.40,13.60,13.10



Higher CV = more variable segment = primary candidate for process improvement.
Most variable segment overall: Furnace→2nd


## 4. Deviation from reference per piece

For each piece, compute the deviation from its die matrix reference at each stage. Positive deviation = slower than reference. This allows identifying both slow individual pieces and systematic drifts.

In [5]:
# Compute per-matrix median reference for each partial
ref = df.groupby('die_matrix')[PARTIAL_COLS].median()

# Merge reference back and compute deviation
df2 = df.copy()
for col in PARTIAL_COLS:
    dev_col = col.replace('partial_', 'dev_')
    df2[dev_col] = df2[col] - df2['die_matrix'].map(ref[col])

DEV_COLS = [col.replace('partial_', 'dev_') for col in PARTIAL_COLS]

print('Deviation statistics (actual - reference, seconds):')
dev_stats = df2.groupby('die_matrix')[DEV_COLS].agg(['mean','std','median']).round(2)
dev_stats.columns = [f'{c[0].replace("dev_","").replace("_s","")}_{c[1]}' for c in dev_stats.columns]
display(dev_stats)
print('\nMedian deviation should be ~0 by construction (reference is the median).')
print('Positive mean deviation indicates right-skewed distribution (more slow pieces than fast).')


Deviation statistics (actual - reference, seconds):


,furnace_to_2ndtrike_mean,furnace_to_2ndtrike_std,furnace_to_2ndtrike_median,2nd_to_3rdtrike_mean,2nd_to_3rdtrike_std,2nd_to_3rdtrike_median,3rd_to_4thtrike_mean,3rd_to_4thtrike_std,3rd_to_4thtrike_median,4thtrike_to_auxiliary_press_mean,4thtrike_to_auxiliary_press_std,4thtrike_to_auxiliary_press_median,auxiliary_press_to_bath_mean,auxiliary_press_to_bath_std,auxiliary_press_to_bath_median
die_matrix,,,,,,,,,,,,,,,
4974,0.46,1.58,0.00,0.07,0.27,0.00,0.10,0.31,0.00,0.06,0.88,0.00,-0.04,0.05,0.00
5052,0.31,1.91,0.00,0.05,0.50,0.00,0.25,0.85,0.00,0.04,1.15,0.00,0.04,0.05,0.00
5090,0.79,2.40,0.00,0.16,0.74,0.00,0.21,1.12,0.00,-0.08,1.62,0.00,0.02,0.09,0.00
5091,0.66,2.42,0.00,-0.00,0.74,0.00,0.26,1.22,0.00,0.04,1.04,0.00,0.04,0.08,0.00



Median deviation should be ~0 by construction (reference is the median).
Positive mean deviation indicates right-skewed distribution (more slow pieces than fast).


## 5. Identify slow pieces and their penalized segment

A piece is considered **slow** if its total bath time exceeds the 90th percentile for its die matrix. For each slow piece, identify which segment contributed the most delay.

In [6]:
# 90th percentile bath time per matrix
p90 = df.groupby('die_matrix')['lifetime_bath_s'].quantile(0.90)
df2['is_slow'] = df2['lifetime_bath_s'] > df2['die_matrix'].map(p90)

print('90th percentile bath time per matrix:')
print(p90.round(1))
print(f'\nSlow pieces: {df2["is_slow"].sum():,} ({100*df2["is_slow"].mean():.1f}%)')

# For each slow piece, find the segment with max positive deviation
slow = df2[df2['is_slow']].copy()
slow['penalized_segment'] = slow[DEV_COLS].idxmax(axis=1).str.replace('dev_','').str.replace('_s','')

print('\nSample of slow pieces with penalized segment:')
display(slow[['timestamp','die_matrix','lifetime_bath_s','penalized_segment']].head(10))


90th percentile bath time per matrix:
die_matrix
4974   59.00
5052   61.80
5090   62.90
5091   63.40
Name: lifetime_bath_s, dtype: float64

Slow pieces: 16,473 (9.8%)

Sample of slow pieces with penalized segment:


,timestamp,die_matrix,lifetime_bath_s,penalized_segment
114,2025-11-06 15:56:48.815000+00:00,5052,67.10,4thtrike_to_auxiliary_press
225,2025-11-06 16:59:53.995000+00:00,5052,63.70,4thtrike_to_auxiliary_press
235,2025-11-06 17:02:37.141000+00:00,5052,64.70,4thtrike_to_auxiliary_press
938,2025-11-06 20:06:41.332000+00:00,5052,67.10,4thtrike_to_auxiliary_press
956,2025-11-06 20:21:07.831000+00:00,5052,62.00,2nd_to_3rdtrike
1068,2025-11-06 23:05:02.464000+00:00,5052,62.60,furnace_to_2ndtrike
1071,2025-11-06 23:05:43.489000+00:00,5052,62.00,furnace_to_2ndtrike
1073,2025-11-06 23:06:10.819000+00:00,5052,62.30,furnace_to_2ndtrike
1075,2025-11-06 23:06:37.537000+00:00,5052,62.60,furnace_to_2ndtrike
1076,2025-11-06 23:06:51.043000+00:00,5052,62.20,furnace_to_2ndtrike


## 6. Slow pieces per die matrix

How slow pieces distribute across die matrices, and which segments are most often penalized per matrix.

In [7]:
total_per_matrix = df2.groupby('die_matrix').size().rename('total')
slow_count = slow.groupby('die_matrix').size().rename('n_slow')
slow_summary = pd.concat([total_per_matrix, slow_count], axis=1).fillna(0)
slow_summary['n_slow'] = slow_summary['n_slow'].astype(int)
slow_summary['pct_slow'] = (100 * slow_summary['n_slow'] / slow_summary['total']).round(1)
display(slow_summary)

# Which segments are most penalized per matrix
penalized = slow.groupby(['die_matrix','penalized_segment']).size().reset_index(name='count')
penalized_pivot = penalized.pivot(index='penalized_segment', columns='die_matrix', values='count').fillna(0).astype(int)
print('\nSlow pieces by penalized segment × matrix:')
display(penalized_pivot)


,total,n_slow,pct_slow
die_matrix,,,
4974,15531,1527,9.80
5052,20887,2081,10.00
5090,81677,7946,9.70
5091,49584,4919,9.90



Slow pieces by penalized segment × matrix:


die_matrix,4974,5052,5090,5091
penalized_segment,,,,
2nd_to_3rdtrike,4,30,334,388
3rd_to_4thtrike,18,152,1236,270
4thtrike_to_auxiliary_press,177,191,577,125
auxiliary_press_to_bath,0,0,1,59
furnace_to_2ndtrike,1328,1708,5798,4077


## 7. Time evolution — detecting drift

Does the process get slower over time for a given matrix? Plot the daily median bath time per matrix to detect progressive deterioration.

In [8]:
daily = df2.groupby(['date','die_matrix'])['lifetime_bath_s'].median().reset_index()
daily.columns = ['date','die_matrix','median_bath_s']
daily['date'] = pd.to_datetime(daily['date'])
daily['die_matrix'] = daily['die_matrix'].astype(str)

chart = alt.Chart(daily).mark_line(opacity=0.7).encode(
    x=alt.X('date:T', title='Date'),
    y=alt.Y('median_bath_s:Q', title='Median bath time (s)', scale=alt.Scale(zero=False)),
    color=alt.Color('die_matrix:N', title='Die Matrix'),
    tooltip=['date:T','die_matrix:N','median_bath_s:Q']
).properties(title='Daily median bath time per die matrix', width=700, height=280)

chart


alt.Chart(...)

## 8. Segment variability ranking

Across all matrices, which process segment is the most unstable? This is where maintenance and process engineering should focus attention.

In [9]:
drift_rows = []
for matrix, grp in df2.groupby('die_matrix'):
    daily_m = grp.groupby('date')['lifetime_bath_s'].median().reset_index()
    daily_m = daily_m.sort_values('date')
    n_days = len(daily_m)
    if n_days < 6: continue
    quartile = max(1, n_days // 4)
    start_median = daily_m.head(quartile)['lifetime_bath_s'].mean()
    end_median   = daily_m.tail(quartile)['lifetime_bath_s'].mean()
    drift_rows.append({
        'die_matrix': matrix,
        'start_median_s': round(start_median, 2),
        'end_median_s':   round(end_median, 2),
        'drift_s':        round(end_median - start_median, 2),
        'drift_%':        round(100*(end_median - start_median)/start_median, 1)
    })

drift_df = pd.DataFrame(drift_rows)
display(drift_df)
print('\nPositive drift = process getting slower over time (tooling wear / process degradation).')
print('Negative drift = process improving (tooling break-in or improved settings).')


,die_matrix,start_median_s,end_median_s,drift_s,drift_%
0,4974,56.95,55.75,-1.20,-2.10
1,5052,57.43,59.37,1.93,3.40
2,5090,56.56,60.46,3.90,6.90
3,5091,56.39,59.74,3.35,5.90



Positive drift = process getting slower over time (tooling wear / process degradation).
Negative drift = process improving (tooling break-in or improved settings).


## 9. Summary

Key findings from the per-matrix analysis.

In [10]:
print('=' * 60)
print('PER-MATRIX ANALYSIS — KEY FINDINGS')
print('=' * 60)

print('\n1. REFERENCE PROFILES (median bath time)')
for matrix in sorted(df['die_matrix'].unique()):
    med = df[df['die_matrix']==matrix]['lifetime_bath_s'].median()
    print(f'   Matrix {matrix}: {med:.1f}s')

print('\n2. MOST VARIABLE SEGMENT')
print(f'   {most_variable} — highest average CV across matrices')

print('\n3. SLOW PIECES (bath > 90th percentile)')
for matrix in sorted(df2['die_matrix'].unique()):
    grp_all  = df2[df2['die_matrix']==matrix]
    grp_slow = slow[slow['die_matrix']==matrix]
    pct = 100 * len(grp_slow) / len(grp_all) if len(grp_all) > 0 else 0
    if len(grp_slow) > 0:
        top_seg = grp_slow['penalized_segment'].value_counts().index[0]
    else:
        top_seg = 'N/A'
    print(f'   Matrix {matrix}: {len(grp_slow):,} slow ({pct:.1f}%), top segment: {top_seg}')

print('\n4. DRIFT (start vs end of active period)')
for _, row in drift_df.iterrows():
    direction = 'SLOWER' if row['drift_s'] > 0 else 'FASTER'
    print(f'   Matrix {int(row["die_matrix"])}: {direction} by {abs(row["drift_s"]):.2f}s ({row["drift_%"]:+.1f}%)')
print('=' * 60)


PER-MATRIX ANALYSIS — KEY FINDINGS

1. REFERENCE PROFILES (median bath time)
   Matrix 4974: 56.0s
   Matrix 5052: 58.3s
   Matrix 5090: 58.1s
   Matrix 5091: 59.1s

2. MOST VARIABLE SEGMENT
   Furnace→2nd — highest average CV across matrices

3. SLOW PIECES (bath > 90th percentile)
   Matrix 4974: 1,527 slow (9.8%), top segment: furnace_to_2ndtrike
   Matrix 5052: 2,081 slow (10.0%), top segment: furnace_to_2ndtrike
   Matrix 5090: 7,946 slow (9.7%), top segment: furnace_to_2ndtrike
   Matrix 5091: 4,919 slow (9.9%), top segment: furnace_to_2ndtrike

4. DRIFT (start vs end of active period)
   Matrix 4974: FASTER by 1.20s (-2.1%)
   Matrix 5052: SLOWER by 1.93s (+3.4%)
   Matrix 5090: SLOWER by 3.90s (+6.9%)
   Matrix 5091: SLOWER by 3.35s (+5.9%)
